# API benchmark: GSM8K / AG News on the Mistral API + EcoLogits
Run the local notebook for a task first: it creates `results/item_ids_<task>.json`, which this notebook reuses.

Setup once in the project folder:
```
pip install mistralai "ecologits[mistralai]"
echo "MISTRAL_API_KEY=your_key" > .env
echo ".env" >> .gitignore
```

In [ ]:
# Config
import os, json, re, time
from pathlib import Path
from datetime import datetime

import pandas as pd
from datasets import load_dataset
from ecologits import EcoLogits
from mistralai import Mistral

# API key: read from a .env file in project folder, otherwise ask once.
def load_key(name="MISTRAL_API_KEY", env_file=Path(".env")):
    if name in os.environ:
        return os.environ[name]
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            k, _, v = line.strip().partition("=")
            if k.strip() == name and v.strip():
                os.environ[name] = v.strip().strip('"').strip("'")
                return os.environ[name]
    from getpass import getpass
    os.environ[name] = getpass(f"{name}: ")
    return os.environ[name]

# EcoLogits must be initialised BEFORE client is created
EcoLogits.init(providers=["mistralai"], electricity_mix_zone="FRA")
client = Mistral(api_key=load_key())

TASK = "agnews"                     # "gsm8k" or "agnews"
# Pinned model IDs so the exact version is reproducible
MODELS = ["ministral-3b-2512", "mistral-large-2512", "mistral-large-4-0"]
MAX_TOKENS_OVERRIDE = None         # e.g. 1024 if large models get truncated; None = task default
SEED = 42
SLEEP_S = 3                      # raise if you get 429 rate limit errors
OUT_DIR = Path("results")
OUT_DIR.mkdir(exist_ok=True)

In [13]:
# Which models can this key use?
sorted(m.id for m in client.models.list().data)

['codestral-2508',
 'codestral-embed',
 'codestral-embed-2505',
 'codestral-latest',
 'deepseek-v4-1-flash-1017',
 'glm-5-2',
 'magistral-medium-latest',
 'magistral-small-latest',
 'ministral-14b-2512',
 'ministral-14b-latest',
 'ministral-3b-2512',
 'ministral-3b-latest',
 'ministral-8b-2512',
 'ministral-8b-latest',
 'mistral-code-fim-latest',
 'mistral-code-latest',
 'mistral-embed',
 'mistral-embed-2312',
 'mistral-large-2512',
 'mistral-large-4',
 'mistral-large-4-0',
 'mistral-large-latest',
 'mistral-medium',
 'mistral-medium-2604',
 'mistral-medium-3',
 'mistral-medium-3-5',
 'mistral-medium-3.5',
 'mistral-medium-latest',
 'mistral-moderation-2603',
 'mistral-ocr-2512',
 'mistral-ocr-3',
 'mistral-ocr-3-0',
 'mistral-ocr-4',
 'mistral-ocr-4-1',
 'mistral-ocr-latest',
 'mistral-small-2603',
 'mistral-small-latest',
 'mistral-vibe-cli-fast',
 'mistral-vibe-cli-latest',
 'mistral-vibe-cli-with-tools',
 'voxtral-mini-2602',
 'voxtral-mini-latest',
 'voxtral-mini-realtime-2602',
 

In [ ]:
# Task definitions: loading, prompt, answer extraction, scoring
NUM_RE = re.compile(r"-?\d[\d,]*\.?\d*")
AG_LABELS = ["World", "Sports", "Business", "Sci/Tech"]

def gsm8k_extract(text):
    # Prefer "####" marker, fall back to last number in text
    if "####" in text:
        m = NUM_RE.search(text.split("####")[-1])
        if m:
            return m.group().replace(",", "").rstrip(".")
    nums = NUM_RE.findall(text)
    return nums[-1].replace(",", "").rstrip(".") if nums else None

def gsm8k_correct(pred, gold):
    try:
        return pred is not None and abs(float(pred) - float(gold)) < 1e-6
    except ValueError:
        return False

def agnews_extract(text):
    # Take first label mentioned in answer
    t = text.lower()
    aliases = {
        "World": ["world"],
        "Sports": ["sports", "sport"],
        "Business": ["business"],
        "Sci/Tech": ["sci/tech", "sci-tech", "science", "technology", "tech"],
    }
    hits = []
    for label, words in aliases.items():
        pos = [t.find(w) for w in words if t.find(w) >= 0]
        if pos:
            hits.append((min(pos), label))
    return min(hits)[1] if hits else None

TASKS = {
    "gsm8k": {
        "load": lambda: load_dataset("openai/gsm8k", "main", split="test"),
        "prompt": (
            "Solve the following math problem step by step. "
            "At the end, write the final answer as a single number on its own line "
            "in the format: #### <number>\n\nProblem: {question}"
        ),
        "question": lambda item: item["question"],
        "gold": lambda item: item["answer"].split("####")[-1].strip().replace(",", ""),
        "extract": gsm8k_extract,
        "correct": gsm8k_correct,
        "max_tokens": 512,
    },
    "agnews": {
        "load": lambda: load_dataset("fancyzhx/ag_news", split="test"),
        "prompt": (
            "Classify the following news article into exactly one of these categories: "
            "World, Sports, Business, Sci/Tech. "
            "Answer with the category name only.\n\nArticle: {question}"
        ),
        "question": lambda item: item["text"],
        "gold": lambda item: AG_LABELS[item["label"]],
        "extract": agnews_extract,
        "correct": lambda pred, gold: pred == gold,
        "max_tokens": 10,
    },
}

T = TASKS[TASK]
MAX_TOKENS = T["max_tokens"]
PROMPT = T["prompt"]

In [ ]:
# Same items as local run
if MAX_TOKENS_OVERRIDE:
    MAX_TOKENS = MAX_TOKENS_OVERRIDE
idx = json.loads((OUT_DIR / f"item_ids_{TASK}.json").read_text())
ds = T["load"]()
items = [ds[i] | {"idx": i} for i in idx]
print(len(items), "items, max tokens:", MAX_TOKENS)

100 items, max tokens: 10


In [ ]:
# Read EcoLogits values (either a number or a min/max range)
def rng(v):
    v = getattr(v, "value", v)
    return (v.min, v.max) if hasattr(v, "min") else (v, v)

ECO_KEYS = ["eco_kwh_min", "eco_kwh_max",
            "eco_gwp_usage_kg_min", "eco_gwp_usage_kg_max",
            "eco_gwp_total_kg_min", "eco_gwp_total_kg_max"]

def ask(model, question, retries=5):
    for attempt in range(retries):
        try:
            t0 = time.perf_counter()
            r = client.chat.complete(
                model=model,
                messages=[{"role": "user", "content": PROMPT.format(question=question)}],
                temperature=0, top_p=1, max_tokens=MAX_TOKENS, random_seed=SEED,
            )
            latency = time.perf_counter() - t0
            break
        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"Error ({e}), retrying in {wait} s")
            time.sleep(wait)
    else:
        raise RuntimeError("Too many failed attempts")

    content = r.choices[0].message.content
    if isinstance(content, list):
        # Reasoning models return chunks; keep only final text, drop thinking part
        texts = [getattr(c, "text", "") for c in content if getattr(c, "type", "") == "text"]
        has_thinking = any(getattr(c, "type", "") == "thinking" for c in content)
        content = "".join(texts)
    else:
        has_thinking = False

    out = {
        "served_model": r.model,
        "output": content or "",
        "has_thinking": has_thinking,
        "input_tokens": r.usage.prompt_tokens,
        "output_tokens": r.usage.completion_tokens,
        "latency_s": latency,
    }

    imp = r.impacts
    if imp is None or imp.usage is None:
        return out | {k: None for k in ECO_KEYS}
    e_lo, e_hi = rng(imp.usage.energy)
    gu_lo, gu_hi = rng(imp.usage.gwp)
    g_lo, g_hi = rng(imp.gwp)
    return out | {"eco_kwh_min": e_lo, "eco_kwh_max": e_hi,
                  "eco_gwp_usage_kg_min": gu_lo, "eco_gwp_usage_kg_max": gu_hi,
                  "eco_gwp_total_kg_min": g_lo, "eco_gwp_total_kg_max": g_hi}

# Smoke test: are impacts filled in, does model reason, which version answers?
for m in MODELS:
    t = ask(m, T["question"](items[0]))
    print(m, {k: v for k, v in t.items() if k != "output"})

ministral-3b-2512 {'served_model': 'ministral-3b-2512', 'has_thinking': False, 'input_tokens': 79, 'output_tokens': 2, 'latency_s': 0.2403958880022401, 'eco_kwh_min': 2.79598130981247e-07, 'eco_kwh_max': 2.79598130981247e-07, 'eco_gwp_usage_kg_min': 1.1586546547862875e-08, 'eco_gwp_usage_kg_max': 1.1586546547862875e-08, 'eco_gwp_total_kg_min': 5.0634896645540435e-08, 'eco_gwp_total_kg_max': 5.0634896645540435e-08}
mistral-large-2512 {'served_model': 'mistral-large-2512', 'has_thinking': False, 'input_tokens': 79, 'output_tokens': 2, 'latency_s': 0.6762925049988553, 'eco_kwh_min': 2.0783540063606297e-05, 'eco_kwh_max': 2.0783540063606297e-05, 'eco_gwp_usage_kg_min': 8.612699002358449e-07, 'eco_gwp_usage_kg_max': 8.612699002358449e-07, 'eco_gwp_total_kg_min': 4.318477107443052e-06, 'eco_gwp_total_kg_max': 4.318477107443052e-06}
mistral-large-4-0 {'served_model': 'mistral-large-4-0', 'has_thinking': True, 'input_tokens': 87, 'output_tokens': 10, 'latency_s': 0.6489277169966954, 'eco_kwh_m

In [ ]:
# Main run: saves after every item && resumes if interrupted (thanks to TU Wifi)
summaries = []
for model in MODELS:
    item_path = OUT_DIR / f"items_{TASK}_api_{model}.csv"
    done = pd.read_csv(item_path) if item_path.exists() else pd.DataFrame()
    done_ids = set(done.idx) if len(done) else set()

    for k, item in enumerate(items):
        if item["idx"] in done_ids:
            continue
        res = ask(model, T["question"](item))
        gold = T["gold"](item)
        pred = T["extract"](res["output"])
        row = {"idx": item["idx"], "gold": gold, "pred": pred,
               "correct": T["correct"](pred, gold),
               "truncated": res["output_tokens"] >= MAX_TOKENS,
               "timestamp": datetime.now().isoformat(), **res}
        pd.DataFrame([row]).to_csv(item_path, mode="a", header=not item_path.exists(), index=False)
        if k % 10 == 0:
            print(f"{model} {k}/{len(items)}")
        time.sleep(SLEEP_S)

    df = pd.read_csv(item_path)
    n_correct = max(int(df.correct.sum()), 1)
    out_tokens = max(int(df.output_tokens.sum()), 1)
    summaries.append({
        "task": TASK, "model": model, "setting": "api",
        "served_model": df.served_model.iloc[0],
        "run_start": df.timestamp.min(), "start_hour": pd.to_datetime(df.timestamp.min()).hour,
        "max_tokens": MAX_TOKENS,
        "n": len(df), "accuracy": df.correct.mean(), "n_correct": int(df.correct.sum()),
        "truncated_share": df.truncated.mean(),
        "thinking_share": df.has_thinking.mean(),
        "mean_input_tokens": df.input_tokens.mean(),
        "mean_output_tokens": df.output_tokens.mean(),
        "mean_latency_s": df.latency_s.mean(),
        "eco_wh_per_query_min": df.eco_kwh_min.mean() * 1000,
        "eco_wh_per_query_max": df.eco_kwh_max.mean() * 1000,
        "eco_wh_per_correct_max": df.eco_kwh_max.sum() * 1000 / n_correct,
        "eco_wh_per_1k_tokens_max": df.eco_kwh_max.sum() * 1000 / out_tokens * 1000,
        "eco_g_co2_usage_per_query_max": df.eco_gwp_usage_kg_max.mean() * 1000,
        "eco_g_co2_total_per_query_max": df.eco_gwp_total_kg_max.mean() * 1000,
    })

api = pd.DataFrame(summaries)
api_path = OUT_DIR / "runs_api.csv"
api.to_csv(api_path, mode="a", header=not api_path.exists(), index=False)
api

mistral-large-4-0 0/100
mistral-large-4-0 10/100
mistral-large-4-0 20/100
mistral-large-4-0 30/100
mistral-large-4-0 40/100
mistral-large-4-0 50/100
mistral-large-4-0 60/100
mistral-large-4-0 70/100
mistral-large-4-0 80/100
Error (API error occurred: Status 504. Body: {"object":"error","message":"Service unavailable.","type":"internal_server_error","param":null,"code":"3804","raw_status_code":504}), retrying in 5 s
mistral-large-4-0 90/100


,task,model,setting,served_model,run_start,start_hour,max_tokens,n,accuracy,n_correct,...,thinking_share,mean_input_tokens,mean_output_tokens,mean_latency_s,eco_wh_per_query_min,eco_wh_per_query_max,eco_wh_per_correct_max,eco_wh_per_1k_tokens_max,eco_g_co2_usage_per_query_max,eco_g_co2_total_per_query_max
0,agnews,ministral-3b-2512,api,ministral-3b-2512,2026-11-18T17:04:07.577922,17,10,100,0.59,59,...,0.0,87.42,2.15,0.331925,0.000305,0.000305,0.000518,0.142086,0.000013,0.000056
1,agnews,mistral-large-2512,api,mistral-large-2512,2026-11-18T17:07:11.053360,17,10,100,0.90,90,...,0.0,87.42,2.54,0.726582,0.020080,0.020080,0.022311,7.905492,0.000832,0.003862
2,agnews,mistral-large-4-0,api,mistral-large-4-0,2026-11-19T10:42:17.164059,10,10,100,0.00,0,...,1.0,98.93,10.00,5.727194,0.050063,0.050063,5.006328,5.006328,0.002075,0.007253


In [ ]:
# Local + API side by side for this task
cols = ["task", "setting", "model", "n", "accuracy", "mean_output_tokens", "mean_latency_s",
        "net_wh_per_query", "gross_wh_per_query", "eco_wh_per_query_min", "eco_wh_per_query_max"]

frames = []
if (OUT_DIR / "runs_local.csv").exists():
    frames.append(pd.read_csv(OUT_DIR / "runs_local.csv"))
if (OUT_DIR / "runs.csv").exists():
    # Older GSM8K run from first notebook version: add missing columns
    old = pd.read_csv(OUT_DIR / "runs.csv")
    old = old.assign(task="gsm8k", setting="local", net_wh_per_query=old["wh_per_query"])
    frames.append(old)
if (OUT_DIR / "runs_api.csv").exists():
    frames.append(pd.read_csv(OUT_DIR / "runs_api.csv"))

both = pd.concat([f.reindex(columns=cols) for f in frames], ignore_index=True)
both[both.task == TASK]

,task,setting,model,n,accuracy,mean_output_tokens,mean_latency_s,net_wh_per_query,gross_wh_per_query,eco_wh_per_query_min,eco_wh_per_query_max
1,agnews,local,ministral-3:3b,100,0.48,2.08,1.924745,0.002055,0.024326,0.000168,0.000168
2,agnews,local,ministral-3:3b,100,0.46,2.16,1.914411,0.002085,0.024295,0.000174,0.000174
7,agnews,api,ministral-3b-2512,100,0.59,2.15,0.331925,NaN,NaN,0.000305,0.000305
8,agnews,api,mistral-large-2512,100,0.90,2.54,0.726582,NaN,NaN,0.020080,0.020080
9,agnews,api,mistral-large-4-0,100,0.00,10.00,2.268319,NaN,NaN,0.049032,0.049032
10,agnews,api,ministral-3b-2512,100,0.59,2.15,0.331925,NaN,NaN,0.000305,0.000305
11,agnews,api,mistral-large-2512,100,0.90,2.54,0.726582,NaN,NaN,0.020080,0.020080
12,agnews,api,mistral-large-4-0,100,0.00,10.00,2.268319,NaN,NaN,0.049032,0.049032
13,agnews,api,ministral-3b-2512,100,0.59,2.15,0.331925,NaN,NaN,0.000305,0.000305
14,agnews,api,mistral-large-2512,100,0.90,2.54,0.726582,NaN,NaN,0.020080,0.020080
